# Import Fidelity — is the HATS data identical to the butler parquet files?

Verifies that the imported HATS catalogs carry exactly the data of the butler-exported
parquet files, at three levels of increasing cost:

1. **Global footer aggregates** (full coverage, reads no data pages): per shared column —
   total value count, null count, global min/max — summed over every butler file footer
   and compared against the HATS catalog's `_metadata`. Catches lost/duplicated rows and
   any corruption that moves an extremum or a null count.
2. **Row-exact sampled partitions**: for K random HATS partitions, find the butler files
   whose footer ra/dec ranges overlap the pixel, read them, compute `_healpix_29` exactly
   as the import did, select the pixel's rows, re-apply the dimension columns from the
   butler index CSVs, sort both sides identically, and compare cell by cell.
3. **Full spatial-assignment map-reduce** (optional, heavy): read *only* ra/dec from every
   butler file and histogram rows onto the catalog's actual partitioning — proves every
   row landed in the right partition, without comparing values.

What is *expected* to differ: HATS-side additions (`_healpix_29`, `*Mag*` columns and
MJD columns from postprocess, dimension columns like `tract`/`patch`/`band` merged from the butler dataIds). Level 1/2 compare the column intersection and report the two
"only on one side" lists so unexpected asymmetries are visible.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import hats
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from hats import pixel_math
from hats.pixel_math.spatial_index import SPATIAL_INDEX_COLUMN, SPATIAL_INDEX_ORDER
from lsst.daf.butler import Butler
from tqdm.auto import tqdm

# flat imported catalogs from the dp2 run; butler queried live (same repo/collection the
# butler stage exported from)
hats_dir = Path("/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2")
collections_dir = Path("/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2-fixed")
REPO = "dp2_prep"
BUTLER_COLLECTION = "LSSTCam/runs/DRP/DP2"

# ra/dec and the dimension columns follow each catalog's config (import_args / dims).
CATALOGS = {
    "dia_object": dict(ra="ra", dec="dec", ids=["diaObjectId"], dims=["tract"]),
    "dia_source": dict(ra="ra", dec="dec", ids=["diaSourceId"], dims=["tract"]),
    "dia_object_forced_source": dict(
        ra="coord_ra", dec="coord_dec", ids=["diaObjectId", "visit", "detector"], dims=["patch", "tract"]
    ),
    "object": dict(ra="coord_ra", dec="coord_dec", ids=["objectId"], dims=["tract"]),
    "source": dict(
        ra="ra", dec="dec", ids=["sourceId"], dims=["band", "day_obs", "physical_filter", "visit"]
    ),
    "object_forced_source": dict(
        ra="coord_ra", dec="coord_dec", ids=["objectId", "visit", "detector"], dims=["patch", "tract"]
    ),
}

N_WORKERS = 16

_butler = None
_ref_cache = {}


def get_butler():
    global _butler
    if _butler is None:
        b = Butler(REPO)
        collections = list(b.registry.queryCollections(BUTLER_COLLECTION))
        _butler = Butler(REPO, collections=collections)
    return _butler


def _butler_refs(catalog_name):
    """URI list + per-file dimension values, straight from the butler (mirrors the butler
    stage: query_datasets -> getManyURIs; dims restricted to the catalog's configured
    `dims`, which is what the import's index files carried)."""
    if catalog_name not in _ref_cache:
        butler = get_butler()
        refs = butler.query_datasets(catalog_name, limit=None)
        uris = butler._datastore.getManyURIs(refs)
        keep = CATALOGS[catalog_name]["dims"]
        paths, dims = [], {}
        for ref, uri in uris.items():
            p = uri.primaryURI.geturl()
            paths.append(p)
            dims[p] = {k: v for k, v in ref.dataId.mapping.items() if k in keep}
        _ref_cache[catalog_name] = (paths, dims)
    return _ref_cache[catalog_name]


def butler_files(catalog_name):
    return _butler_refs(catalog_name)[0]


def index_dimensions(catalog_name):
    return _butler_refs(catalog_name)[1]

## Level 1 — global footer aggregates

One footer read per butler file (threaded; the forced-source catalogs have ~100k files,
so expect minutes) against one `_metadata` read on the HATS side. Comparison is over the
column intersection; extra columns on either side are listed, not failed — check the
lists against the expected additions.

In [ ]:
def aggregate_footers(metadata_iter):
    """path -> [num_values, null_count, min, max, stats_complete] summed over row groups."""
    total_rows = 0
    agg = {}
    for md_ in metadata_iter:
        total_rows += md_.num_rows
        for rg_i in range(md_.num_row_groups):
            rg = md_.row_group(rg_i)
            for c_i in range(rg.num_columns):
                col = rg.column(c_i)
                rec = agg.setdefault(col.path_in_schema, [0, 0, None, None, True, col.physical_type])
                rec[0] += col.num_values
                st = col.statistics
                if st is None or st.null_count is None:
                    rec[4] = False
                else:
                    rec[1] += st.null_count
                    if st.has_min_max:
                        rec[2] = st.min if rec[2] is None else min(rec[2], st.min)
                        rec[3] = st.max if rec[3] is None else max(rec[3], st.max)
                    else:
                        rec[4] = False
    return total_rows, agg


def read_footer(path):
    from lsst.resources import ResourcePath

    with ResourcePath(path).open("rb") as f:
        return pq.read_metadata(f)


_butler_agg_cache = {}


def butler_aggregates(catalog_name):
    if catalog_name not in _butler_agg_cache:
        files = butler_files(catalog_name)
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            footers = tqdm(ex.map(read_footer, files), total=len(files),
                           desc=f"{catalog_name} footers", unit="file")
            _butler_agg_cache[catalog_name] = aggregate_footers(footers)
    else:
        # upgrade pre-physical-type cache entries in place (one footer read for the types)
        _, agg = _butler_agg_cache[catalog_name]
        if agg and len(next(iter(agg.values()))) == 5:
            md0 = read_footer(butler_files(catalog_name)[0])
            types = {}
            for rg_i in range(md0.num_row_groups):
                rg = md0.row_group(rg_i)
                for c_i in range(rg.num_columns):
                    types.setdefault(rg.column(c_i).path_in_schema, rg.column(c_i).physical_type)
            for path, rec in agg.items():
                rec.append(types.get(path))
    return _butler_agg_cache[catalog_name]


def minmax_equal(b, h):
    """Exact min/max agreement, allowing the postprocess stage's deliberate
    float64 -> float32 downcast (rounding is monotonic, so min/max survive the
    cast exactly: compare butler values cast to float32, bit-for-bit)."""
    if (b[2], b[3]) == (h[2], h[3]):
        return True
    if b[5] == "DOUBLE" and h[5] == "FLOAT":
        return (np.float32(b[2]), np.float32(b[3])) == (np.float32(h[2]), np.float32(h[3]))
    return False


def compare_catalog_aggregates(catalog_name):
    print(f"===== {catalog_name} =====")
    hats_path = hats_dir / catalog_name
    if not hats_path.exists():
        print("  SKIP — hats catalog not found")
        return

    files = butler_files(catalog_name)
    rows_b, butler = butler_aggregates(catalog_name)
    rows_h, hats_agg = aggregate_footers([pq.read_metadata(hats_path / "dataset" / "_metadata")])

    print(f"  butler: {len(files):,} files, {rows_b:,} rows | hats: {rows_h:,} rows "
          f"{'OK' if rows_b == rows_h else '*** ROW COUNTS DIFFER ***'}")

    shared = sorted(set(butler) & set(hats_agg))
    dims = set(index_dimensions(catalog_name)[files[0]].keys()) if files else set()
    hats_only = sorted(set(hats_agg) - set(butler))
    butler_only = sorted(set(butler) - set(hats_agg))
    unexpected = [c for c in hats_only
                  if c != SPATIAL_INDEX_COLUMN and "Mag" not in c and "Mjd" not in c and c not in dims]
    print(f"  shared columns: {len(shared)}; hats-only: {len(hats_only)} "
          f"(unexpected: {unexpected or 'none'}); butler-only: {butler_only or 'none'}")

    diffs = []
    for col in shared:
        b, h = butler[col], hats_agg[col]
        if b[0] != h[0]:
            diffs.append(f"{col}: total values {b[0]:,} vs {h[0]:,}")
        if b[4] and h[4]:
            if b[1] != h[1]:
                diffs.append(f"{col}: nulls {b[1]:,} vs {h[1]:,}")
            if not minmax_equal(b, h):
                diffs.append(f"{col}: min/max ({b[2]}, {b[3]}) vs ({h[2]}, {h[3]})")
    downcast = [c for c in shared if butler[c][5] == "DOUBLE" and hats_agg[c][5] == "FLOAT"]
    if downcast:
        print(f"  {len(downcast)} column(s) float64->float32 downcast by postprocess (expected; "
              "min/max compared after cast)")
    if diffs or rows_b != rows_h:
        print(f"  *** {len(diffs)} column-aggregate difference(s):")
        for d in diffs[:20]:
            print("     ", d)
    else:
        print("  ALL SHARED-COLUMN AGGREGATES IDENTICAL")

## Cache persistence

Everything expensive in this notebook accumulates in three dicts: `_ref_cache` (butler
URI queries), `_butler_agg_cache` (the ~100k-file footer sweeps), and `_range_cache`
(the level-2 ra/dec footer ranges). Run the save cell after any big computation; run the
load cell after a kernel restart to pick up where you left off. The cache is keyed by
catalog name only — **delete the file if the underlying catalogs change** (e.g. after
the missing-objects patch), or the checks will compare against stale aggregates.

In [ ]:
import pickle

CACHE_FILE = Path("/sdf/data/rubin/shared/lsdb_commissioning/validation/dp2-fixed/notebook12_cache.pkl")
_range_cache = {}  # catalog -> {path: (ra_range, dec_range)} for the level-2 scans


def save_verification_cache():
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, "wb") as f:
        pickle.dump({"refs": _ref_cache, "aggs": _butler_agg_cache, "ranges": _range_cache}, f)
    print(f"saved: refs={list(_ref_cache)}, aggs={list(_butler_agg_cache)}, "
          f"ranges={list(_range_cache)} -> {CACHE_FILE}")


def load_verification_cache():
    if not CACHE_FILE.exists():
        print("no cache file at", CACHE_FILE)
        return
    with open(CACHE_FILE, "rb") as f:
        data = pickle.load(f)
    _ref_cache.update(data.get("refs", {}))
    _butler_agg_cache.update(data.get("aggs", {}))
    _range_cache.update(data.get("ranges", {}))
    print(f"loaded: refs={list(_ref_cache)}, aggs={list(_butler_agg_cache)}, ranges={list(_range_cache)}")


load_verification_cache()

In [ ]:
for name in CATALOGS:
    compare_catalog_aggregates(name)

## Level 2 — row-exact comparison on sampled partitions

For each sampled HATS partition: butler files are pruned by their footer ra/dec ranges
against the pixel's corner-derived bounding box (padded; a superset is fine — exact
membership is decided by `_healpix_29`), matching rows are selected by computing the
spatial index exactly as the import did, dimension columns are re-applied from the butler dataIds, and both sides are sorted by the catalog's id columns before an exact per-column
comparison.

In [ ]:
from cdshealpix.nested import vertices as healpix_vertices
from lsst.resources import ResourcePath


def pixel_bbox(order, pixel, pad_deg=0.1):
    lon, lat = healpix_vertices(np.asarray([pixel], dtype=np.uint64), order)
    ras, decs = lon.deg.ravel(), lat.deg.ravel()
    if ras.max() - ras.min() > 180:  # pixel wraps RA=0 — skip RA pruning for safety
        return None, (decs.min() - pad_deg, decs.max() + pad_deg)
    return (ras.min() - pad_deg, ras.max() + pad_deg), (decs.min() - pad_deg, decs.max() + pad_deg)


def footer_ra_dec_range(path, ra_col, dec_col):
    md_ = read_footer(path)
    ra = [None, None]
    dec = [None, None]
    for rg_i in range(md_.num_row_groups):
        rg = md_.row_group(rg_i)
        for c_i in range(rg.num_columns):
            col = rg.column(c_i)
            tgt = ra if col.path_in_schema == ra_col else dec if col.path_in_schema == dec_col else None
            if tgt is not None and col.statistics is not None and col.statistics.has_min_max:
                st = col.statistics
                tgt[0] = st.min if tgt[0] is None else min(tgt[0], st.min)
                tgt[1] = st.max if tgt[1] is None else max(tgt[1], st.max)
    return ra, dec


def overlaps(file_range, box):
    if box is None or file_range[0] is None:
        return True  # no pruning possible — keep the file
    lo, hi = box
    return file_range[1] >= lo and file_range[0] <= hi


def verify_partition(catalog_name, order, pixel, file_ranges, dims):
    cfg = CATALOGS[catalog_name]
    ra_box, dec_box = pixel_bbox(order, pixel)
    candidates = [p for p, (ra, dec) in file_ranges.items() if overlaps(ra, ra_box) and overlaps(dec, dec_box)]

    # butler side: read candidates, compute the spatial index, select the pixel's rows
    shift = 2 * (SPATIAL_INDEX_ORDER - order)
    parts = []
    for path in candidates:
        with ResourcePath(path).open("rb") as f:
            tbl = pq.ParquetFile(f).read()
        if len(tbl) == 0:
            continue
        h29 = pixel_math.compute_spatial_index(
            tbl[cfg["ra"]].to_numpy().astype(np.float64), tbl[cfg["dec"]].to_numpy().astype(np.float64)
        )
        mask = (h29 >> shift) == pixel
        if mask.any():
            sub = tbl.filter(pa.array(mask)).to_pandas()
            for dim_col, dim_val in dims.get(path, {}).items():
                sub[dim_col] = dim_val
            parts.append(sub)
    butler_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

    # hats side
    hats_file = (hats_dir / catalog_name / "dataset" / f"Norder={order}" /
                 f"Dir={pixel // 10000 * 10000}" / f"Npix={pixel}.parquet")
    hats_df = pq.ParquetFile(hats_file).read().to_pandas()

    label = f"({order}, {pixel})"
    if len(butler_df) != len(hats_df):
        print(f"  {label}: *** ROW COUNT {len(butler_df):,} (butler) vs {len(hats_df):,} (hats)")
        return False
    shared = [c for c in hats_df.columns if c in butler_df.columns]
    left = butler_df.sort_values(cfg["ids"], ignore_index=True)[shared]
    right = hats_df.sort_values(cfg["ids"], ignore_index=True)[shared]
    bad_cols = [c for c in shared if not left[c].equals(right[c])]
    if bad_cols:
        print(f"  {label}: *** {len(hats_df):,} rows, {len(candidates)} files — columns differ: {bad_cols}")
        return False
    print(f"  {label}: OK — {len(hats_df):,} rows x {len(shared)} shared columns identical "
          f"({len(candidates)} butler files)")
    return True


def verify_sampled_partitions(catalog_name, n_partitions=3, seed=42):
    print(f"===== {catalog_name} =====")
    hats_path = hats_dir / catalog_name
    if not hats_path.exists():
        print("  SKIP — hats catalog not found")
        return
    cfg = CATALOGS[catalog_name]
    cat = hats.read_hats(hats_path)
    pixels = cat.get_healpix_pixels()
    rng = np.random.default_rng(seed)
    sample = [pixels[i] for i in rng.choice(len(pixels), size=min(n_partitions, len(pixels)), replace=False)]

    files = butler_files(catalog_name)
    dims = index_dimensions(catalog_name)
    if catalog_name in _range_cache:
        ranges = _range_cache[catalog_name]
    else:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            range_iter = tqdm(ex.map(lambda p: footer_ra_dec_range(p, cfg["ra"], cfg["dec"]), files),
                              total=len(files), desc=f"{catalog_name} ra/dec ranges", unit="file")
            ranges = dict(zip(files, range_iter))
        _range_cache[catalog_name] = ranges

    all_ok = True
    for p in sample:
        all_ok &= verify_partition(catalog_name, p.order, p.pixel, ranges, dims)
    print("  PASS" if all_ok else "  *** FAIL — see above")

In [ ]:
# The object-family catalogs are the cheapest place to start; the source-family ones
# read more butler files per pixel. Tune n_partitions to taste.
for name in ["dia_object", "object"]:
    verify_sampled_partitions(name, n_partitions=3)

In [ ]:
for name in ["dia_source", "dia_object_forced_source", "object_forced_source", "source"]:
    verify_sampled_partitions(name, n_partitions=2)

## Level 3 (optional) — full spatial-assignment map-reduce

Reads *only* ra/dec from every butler file (the "map"), histograms rows onto the HATS
catalog's actual partition list (the "reduce"), and compares per-partition row counts
with the catalog's `_metadata`. Full-coverage proof that every row was assigned to the
right partition — combine with level 1 (values) and level 2 (row-exact samples) for the
complete fidelity argument. Heavy: budget accordingly, or set `sample_fraction < 1`.

In [ ]:
def assignment_histogram(catalog_name, sample_fraction=1.0, seed=0):
    print(f"===== {catalog_name} =====")
    cfg = CATALOGS[catalog_name]
    cat = hats.read_hats(hats_dir / catalog_name)
    # destination pixel per catalog partition, as (order, pixel) -> expected rows from _metadata
    md_ = pq.read_metadata(hats_dir / catalog_name / "dataset" / "_metadata")
    expected = {}
    for rg_i in range(md_.num_row_groups):
        rg = md_.row_group(rg_i)
        expected[rg.column(0).file_path] = expected.get(rg.column(0).file_path, 0) + rg.num_rows
    pixel_expected = {}
    for fp, n in expected.items():
        import re

        o = int(re.search(r"Norder=(\d+)", fp)[1])
        px = int(re.search(r"Npix=(\d+)", fp)[1])
        pixel_expected[(o, px)] = n

    orders = sorted({o for o, _ in pixel_expected})
    files = butler_files(catalog_name)
    if sample_fraction < 1.0:
        rng = np.random.default_rng(seed)
        files = list(rng.choice(files, size=int(len(files) * sample_fraction), replace=False))
        print(f"  SAMPLING {len(files):,} files — counts will not match totals; "
              "use only to smoke-test the machinery")

    def file_counts(path):
        with ResourcePath(path).open("rb") as f:
            tbl = pq.ParquetFile(f).read(columns=[cfg["ra"], cfg["dec"]])
        if len(tbl) == 0:
            return {}
        h29 = pixel_math.compute_spatial_index(
            tbl[cfg["ra"]].to_numpy().astype(np.float64), tbl[cfg["dec"]].to_numpy().astype(np.float64)
        )
        out = {}
        for o in orders:
            px, cnt = np.unique(h29 >> (2 * (SPATIAL_INDEX_ORDER - o)), return_counts=True)
            for p, c in zip(px, cnt):
                if (o, int(p)) in pixel_expected:
                    out[(o, int(p))] = out.get((o, int(p)), 0) + int(c)
        return out

    observed = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        for res in tqdm(ex.map(file_counts, files), total=len(files),
                        desc=f"{catalog_name} assignment", unit="file"):
            for k, v in res.items():
                observed[k] = observed.get(k, 0) + v

    # note: rows in overlapping-order coverage are counted toward every order, so compare per pixel
    bad = {k: (observed.get(k, 0), pixel_expected[k]) for k in pixel_expected
           if observed.get(k, 0) != pixel_expected[k]}
    if sample_fraction < 1.0:
        print(f"  (sampled run — {len(bad)} pixels differ, as expected under sampling)")
    elif bad:
        print(f"  *** {len(bad)} of {len(pixel_expected)} partitions have row-count mismatches:")
        for k, (o_, e_) in list(bad.items())[:10]:
            print(f"      {k}: butler {o_:,} vs hats {e_:,}")
    else:
        print(f"  ALL {len(pixel_expected)} partitions: butler spatial assignment matches exactly")


# assignment_histogram("dia_object")                      # full run
# assignment_histogram("object_forced_source", 0.01)      # machinery smoke-test

## Collections under dp2-fixed — derived-data aggregates vs butler

The collections are derived (flat catalogs -> nested join -> re-tile), so the comparison
maps: **base columns** of the main catalog against the butler object table (exact
equality), and each **nested column's leaves** against the corresponding butler source
table. For nested leaves, exact equality is *not* expected: every object with an empty
source list adds one def-level slot that parquet counts as a null on **every** leaf. The
correct invariant is that the surplus over butler — in both `num_values` and
`null_count` — is the **same constant E on every leaf** of that nested column (E = number
of empty lists), with min/max unchanged. A leaf whose surplus differs from its siblings'
is exactly the corruption signature from notebook 11.

(Row-exact verification of the join itself is out of scope here — butler <-> flat is
covered row-exactly by level 2 above, and flat <-> nested is the nesting stage's
concern, spot-checkable with notebook 11 section 16's slice comparison.)

In [ ]:
def _leaf_map(hats_agg, nested_columns):
    """Split hats leaf paths into base columns and per-nested-column {butler_col: leaf_path}."""
    base, nested = {}, {c: {} for c in nested_columns}
    for path in hats_agg:
        parts = path.split(".")
        if parts[0] in nested:
            nested[parts[0]][path.removeprefix(parts[0] + ".").removesuffix(".list.element")] = path
        else:
            base[path.removesuffix(".list.element")] = path
    return base, nested


def compare_collection_aggregates(collection_name, main_catalog, base_butler, nested_map):
    print(f"===== {collection_name} =====")
    main_path = collections_dir / collection_name / main_catalog
    if not main_path.exists():
        print("  SKIP — not found:", main_path)
        return
    rows_h, hats_agg = aggregate_footers([pq.read_metadata(main_path / "dataset" / "_metadata")])
    base, nested = _leaf_map(hats_agg, list(nested_map))

    # base columns <-> butler object table (exact)
    rows_b, butler = butler_aggregates(base_butler)
    print(f"  rows: butler {base_butler} {rows_b:,} vs main catalog {rows_h:,} "
          f"{'OK' if rows_b == rows_h else '*** DIFFER ***'}")
    diffs = []
    for col, path in sorted(base.items()):
        if col not in butler:
            continue  # pipeline-added base column (Mag, mjd, dims, _healpix_29)
        b, h = butler[col], hats_agg[path]
        if b[0] != h[0]:
            diffs.append(f"base {col}: total values {b[0]:,} vs {h[0]:,}")
        if b[4] and h[4] and b[1] != h[1]:
            diffs.append(f"base {col}: nulls {b[1]:,} vs {h[1]:,}")
        if b[4] and h[4] and not minmax_equal(b, h):
            diffs.append(f"base {col}: min/max ({b[2]}, {b[3]}) vs ({h[2]}, {h[3]})")

    # nested leaves <-> butler source tables (constant-surplus invariant)
    for ncol, src_catalog in nested_map.items():
        _, src_agg = butler_aggregates(src_catalog)
        surpluses, null_surpluses = {}, {}
        for col, path in sorted(nested[ncol].items()):
            if col not in src_agg:
                continue  # pipeline-added source column (Mag, mjd, dims, corrected cols)
            b, h = src_agg[col], hats_agg[path]
            surpluses[col] = h[0] - b[0]
            if b[4] and h[4]:
                null_surpluses[col] = h[1] - b[1]
                if not minmax_equal(b, h):
                    diffs.append(f"{ncol}.{col}: min/max ({b[2]}, {b[3]}) vs ({h[2]}, {h[3]})")
        distinct = set(surpluses.values())
        if len(distinct) == 1:
            E = next(iter(distinct))
            if E >= 0:
                null_bad = {c: s for c, s in null_surpluses.items() if s != E}
                if null_bad:
                    diffs.append(f"{ncol}: null surplus != value surplus ({E:,}) for {null_bad}")
                else:
                    print(f"  {ncol}: {len(surpluses)} leaves vs butler {src_catalog} — consistent "
                          f"(E = {E:,} empty/null entries)")
            else:
                # constant negative = whole source rows uniformly absent — NOT corruption.
                # Causes: orphaned ids (source rows whose object id has no object row — expected
                # to be excluded) and/or margin losses in join_nested (source position more than
                # margin_radius_arcsec outside its object's partition). See the orphan-vs-margin
                # diagnostic cell below.
                diffs.append(f"{ncol}: uniformly MISSING {-E:,} source rows vs butler {src_catalog} "
                             f"(constant across all {len(surpluses)} leaves — dropped rows, not corruption)")
        else:
            worst = sorted(surpluses.items(), key=lambda kv: kv[1])
            diffs.append(f"{ncol}: leaf value surpluses NOT constant — "
                         f"min {worst[0]}, max {worst[-1]} (corruption signature!)")

    if diffs:
        print(f"  *** {len(diffs)} difference(s):")
        for d in diffs[:20]:
            print("     ", d)
    else:
        print("  ALL AGGREGATES CONSISTENT with the butler tables")

In [ ]:
compare_collection_aggregates(
    "dia_object_collection",
    main_catalog="dia_object_lc",
    base_butler="dia_object",
    nested_map={"diaSource": "dia_source", "diaObjectForcedSource": "dia_object_forced_source"},
)
compare_collection_aggregates(
    "object_collection",
    main_catalog="object_lc",
    base_butler="object",
    nested_map={"objectForcedSource": "object_forced_source"},
)

In [ ]:
# The uncertainty-corrected collections must carry the same butler-derived data —
# corrected columns are pipeline additions and are skipped by the mapping automatically.
compare_collection_aggregates(
    "dia_object_collection_uncertainty_corrected",
    main_catalog="dia_object_lc",
    base_butler="dia_object",
    nested_map={"diaSource": "dia_source", "diaObjectForcedSource": "dia_object_forced_source"},
)
compare_collection_aggregates(
    "object_collection_uncertainty_corrected",
    main_catalog="object_lc",
    base_butler="object",
    nested_map={"objectForcedSource": "object_forced_source"},
)

## Diagnosing a uniform source-row deficit: orphans vs margin losses

A constant negative surplus on a nested column means N butler source rows are absent from
the LC catalog. Two causes, with different verdicts:

- **Orphans**: source rows whose object id has no row in the object table (e.g.
  solar-system-associated diaSources linked to an ssObject rather than a diaObject).
  Correctly excluded from an object-keyed LC catalog — document, don't fix.
- **Margin losses**: sources with a valid object, dropped because `join_nested` only sees
  the object's partition + margin cache, and the source's *detection position* lies more
  than `margin_radius_arcsec` (2 arcsec in the dia nesting config) outside that partition.
  Real data loss — fix by raising the margin radius on the next nesting run.

This cell splits the deficit using the flat dp2 catalogs (id columns only, threaded).

In [ ]:
def read_column(catalog_dir, column):
    files = sorted((catalog_dir / "dataset").rglob("*.parquet"))
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        parts = list(tqdm(ex.map(lambda f: pq.read_table(f, columns=[column])[column].to_numpy(), files),
                          total=len(files), desc=f"{catalog_dir.name}.{column}", unit="file"))
    return np.concatenate(parts)


def orphan_vs_margin(source_catalog, object_catalog, join_id, expected_deficit=None):
    src_ids = read_column(hats_dir / source_catalog, join_id)
    obj_ids = read_column(hats_dir / object_catalog, join_id)
    null_ids = int(pd.isna(src_ids).sum()) if src_ids.dtype == object else 0
    orphan = ~np.isin(src_ids, obj_ids)
    n_orphan = int(orphan.sum())
    print(f"{source_catalog}: {len(src_ids):,} rows; orphaned {join_id} (no matching object row): "
          f"{n_orphan:,} ({null_ids:,} null)")
    if expected_deficit is not None:
        n_margin = expected_deficit - n_orphan
        print(f"deficit {expected_deficit:,} = {n_orphan:,} orphans (expected exclusions) "
              f"+ {n_margin:,} margin losses (real loss — raise margin_radius_arcsec if > 0)")
        if n_margin < 0:
            print("*** orphans exceed the deficit — some orphans ARE in the LC catalog? investigate")


orphan_vs_margin("dia_source", "dia_object", "diaObjectId", expected_deficit=8_136_150)

## nested-pandas read sweep — does every partition file actually load?

The footer checks prove structural invariants without reading data; this is the
complementary end-to-end test: load each partition file with `npd.read_parquet`, which
performs the struct -> NestedDtype cast that *enforces* the equal-lengths invariant (the
check that first exposed the coord_dec corruption). A file that passes here will load for
any lsdb/nested-pandas user.

Cost control: `columns=` restricts to e.g. just the nested column (the cast is per
struct column, so that is where the risk lives), and `sample=` spot-checks N random
files instead of all. Reading the source-family catalogs in full is a multi-TB sweep —
sample those unless you have a reason not to.

In [ ]:
import nested_pandas as npd


def npd_read_check(catalog_dir, columns=None, sample=None, seed=0, workers=8):
    catalog_dir = Path(catalog_dir)
    files = sorted((catalog_dir / "dataset").rglob("*.parquet"))
    if sample is not None and sample < len(files):
        rng = np.random.default_rng(seed)
        files = [files[i] for i in sorted(rng.choice(len(files), size=sample, replace=False))]

    def one(f):
        try:
            nf = npd.read_parquet(f, columns=columns)
            return str(f), len(nf), None
        except Exception as e:  # noqa: BLE001 — any load failure is the finding
            return str(f), -1, repr(e)

    with ThreadPoolExecutor(max_workers=workers) as ex:
        results = list(tqdm(ex.map(one, files), total=len(files),
                            desc=f"npd read {catalog_dir.name}", unit="file"))

    failures = [(f, err) for f, _, err in results if err]
    rows = sum(n for _, n, err in results if err is None)
    scope = f"{len(files)} files" + (f" (sample of catalog)" if sample else "")
    print(f"{catalog_dir.name}: {scope}, {rows:,} rows loaded, {len(failures)} failure(s)")
    for f, err in failures[:10]:
        print(f"  *** {f}\n      {err}")
    return failures

In [ ]:
# The nested LC collections — full sweeps (this is where the NestedDtype cast matters).
npd_read_check(collections_dir / "dia_object_collection" / "dia_object_lc")
npd_read_check(collections_dir / "dia_object_collection_uncertainty_corrected" / "dia_object_lc")
npd_read_check(collections_dir / "object_collection" / "object_lc")
npd_read_check(collections_dir / "patched_object" / "object_lc")

In [ ]:
# Flat catalogs — no nested columns, so this only exercises plain loading; sample the
# big source-family ones rather than reading multi-TB catalogs end to end.
npd_read_check(hats_dir / "dia_object")
npd_read_check(hats_dir / "object")
npd_read_check(hats_dir / "dia_source", sample=50)
npd_read_check(hats_dir / "source", sample=50)
npd_read_check(hats_dir / "dia_object_forced_source", sample=50)
npd_read_check(hats_dir / "object_forced_source", sample=50)

## Release readiness — RC folder checks

Everything the release prep added, verified per entry of the RC folder (through the
symlinks, so what's checked is what users will see):

1. **Inventory**: exactly the expected entries, each of the expected kind.
2. **Naming**: `obs_collection` in every `collection.properties` matches the
   release-facing entry name (catches the `patched_object`-style rename wart), and the
   collection's structural references (primary table, margins, indexes) point at member
   dirs that exist.
3. **Default columns**: the main catalog's `hats_cols_default` is present, every entry
   resolves against the schema (dotted nested names included), and the corrected
   collections include the corrected/flag/magErr columns in their defaults.
4. **Static artifacts** by kind — catalog: per-partition stats + skymap/partition PNGs +
   `index.html`/`README.md`; collection: PNGs + summaries; margin/index: summaries only.
5. **lsdb smoke test** (optional, slower): `open_catalog` on each collection — margin
   attaches, and the corrected columns appear in the *default* nested view.

In [ ]:
RC_DIR = Path("/sdf/data/rubin/shared/lsdb_commissioning/hats/dp2_rc1")
RUN_LSDB_SMOKE = True

EXPECTED_ENTRIES = {
    "dia_object": "catalog",
    "dia_source": "catalog",
    "dia_object_forced_source": "catalog",
    "object": "catalog",
    "source": "catalog",
    "object_forced_source": "catalog",
    "dia_object_collection": "collection",
    "object_collection": "collection",
    "dia_object_collection_uncertainty_corrected": "collection",
    "object_collection_uncertainty_corrected": "collection",
}
CORRECTED_DEFAULTS = {
    "dia_object_collection_uncertainty_corrected": ("diaObjectForcedSource", [
        "psfFluxErr_corrected", "psfFluxErr_corrected_flag",
        "psfDiffFluxErr_corrected", "psfDiffFluxErr_corrected_flag", "psfMagErr_corrected",
    ]),
    "object_collection_uncertainty_corrected": ("objectForcedSource", [
        "psfFluxErr_corrected", "psfFluxErr_corrected_flag",
        "psfDiffFluxErr_corrected", "psfDiffFluxErr_corrected_flag", "psfMagErr_corrected",
    ]),
}
STATIC_BY_KIND = {
    "collection": ["skymap.png", "partition_info.png", "index.html", "README.md"],
    "catalog": ["per_partition_statistics.parquet", "skymap.png", "partition_info.png",
                "index.html", "README.md"],
    "margin": ["index.html", "README.md"],
    "index": ["index.html", "README.md"],
}


def read_properties(path):
    props = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            props[k.strip()] = v.strip()
    return props


def entry_kind(path):
    if (path / "collection.properties").exists():
        return "collection"
    ctype = read_properties(path / "properties").get("dataproduct_type", "")
    return ctype if ctype in ("margin", "index") else "catalog"


def schema_leaf_names(catalog_dir):
    schema = pq.read_schema(catalog_dir / "dataset" / "_common_metadata")
    names = set()
    for field in schema:
        names.add(field.name)
        if pa.types.is_struct(field.type):
            for i in range(field.type.num_fields):
                names.add(f"{field.name}.{field.type.field(i).name}")
    return names


def check_static(path, kind, problems, label):
    for artifact in STATIC_BY_KIND[kind]:
        if not (path / artifact).exists():
            problems.append(f"{label}: missing {artifact}")


def check_entry(entry):
    problems = []
    kind = entry_kind(entry)
    expected_kind = EXPECTED_ENTRIES.get(entry.name)
    if expected_kind and kind != expected_kind:
        problems.append(f"{entry.name}: kind {kind!r}, expected {expected_kind!r}")

    if kind != "collection":
        check_static(entry, kind, problems, entry.name)
        return kind, problems

    # -- collection: naming + structural references --
    cprops = read_properties(entry / "collection.properties")
    if cprops.get("obs_collection") != entry.name:
        problems.append(f"{entry.name}: obs_collection={cprops.get('obs_collection')!r} "
                        f"!= entry name (rename wart)")
    main_name = Path(cprops.get("hats_primary_table_url", "")).name
    members = {main_name: "catalog"}
    for key, mkind in (("all_margins", "margin"), ("all_indexes", "index")):
        for token in cprops.get(key, "").replace(",", " ").split():
            members[token.split("=")[-1]] = mkind  # all_indexes tokens can be 'col=name'
    for member, mkind in members.items():
        if not member:
            problems.append(f"{entry.name}: empty {mkind} reference in collection.properties")
        elif not (entry / member / "dataset").is_dir():
            problems.append(f"{entry.name}: referenced {mkind} {member!r} has no dataset dir")

    check_static(entry, "collection", problems, entry.name)
    for member, mkind in members.items():
        if (entry / member).is_dir():
            check_static(entry / member, mkind, problems, f"{entry.name}/{member}")

    # -- default columns on the main catalog --
    main_dir = entry / main_name
    defaults = read_properties(main_dir / "properties").get("hats_cols_default", "")
    defaults = defaults.replace(",", " ").split()
    if not defaults:
        problems.append(f"{entry.name}/{main_name}: no hats_cols_default")
    else:
        leaves = schema_leaf_names(main_dir)
        unresolved = [c for c in defaults if c not in leaves]
        if unresolved:
            problems.append(f"{entry.name}/{main_name}: defaults not in schema: {unresolved}")
        if entry.name in CORRECTED_DEFAULTS:
            ncol, cols = CORRECTED_DEFAULTS[entry.name]
            missing = [f"{ncol}.{c}" for c in cols if f"{ncol}.{c}" not in defaults]
            if missing:
                problems.append(f"{entry.name}/{main_name}: corrected columns absent from "
                                f"defaults: {missing}")
        # margins are loaded with the main's column list — their files must carry it
        for member, mkind in members.items():
            if mkind == "margin" and (entry / member / "dataset").is_dir():
                margin_leaves = schema_leaf_names(entry / member)
                gaps = [c for c in defaults if c not in margin_leaves]
                if gaps:
                    problems.append(f"{entry.name}/{member}: default columns missing from "
                                    f"margin schema: {gaps}")
    return kind, problems


entries = sorted(p for p in RC_DIR.iterdir() if p.is_dir())
names = {e.name for e in entries}
all_problems = [f"MISSING entry: {n}" for n in EXPECTED_ENTRIES if n not in names]
all_problems += [f"UNEXPECTED entry: {n}" for n in sorted(names - set(EXPECTED_ENTRIES))]

for entry in entries:
    kind, problems = check_entry(entry)
    status = "ok" if not problems else f"{len(problems)} problem(s)"
    print(f"{entry.name:50s} [{kind}] {status}")
    all_problems += problems

print()
if all_problems:
    print(f"*** {len(all_problems)} problem(s):")
    for p in all_problems:
        print("  ", p)
else:
    print("RC folder release-ready: inventory, naming, defaults, and static artifacts all check out")

In [ ]:
# lsdb smoke test: open each collection as a user would (default columns), confirm the
# margin attaches and the corrected columns are in the DEFAULT nested view.
import lsdb

if RUN_LSDB_SMOKE:
    for name, kind in EXPECTED_ENTRIES.items():
        if kind != "collection" or not (RC_DIR / name).exists():
            continue
        cat = lsdb.open_catalog(RC_DIR / name)
        margin_ok = cat.margin is not None
        note = f"margin={'attached' if margin_ok else '*** MISSING ***'}, " \
               f"{len(cat.columns)} default columns"
        if name in CORRECTED_DEFAULTS:
            ncol, cols = CORRECTED_DEFAULTS[name]
            fields = list(cat._ddf.meta[ncol].nest.fields)
            missing = [c for c in cols if c not in fields]
            note += f", corrected in default view: {'yes' if not missing else f'*** {missing} ***'}"
        print(f"{name:50s} {note}")

## Notes and caveats

- **Level 2 pruning is a superset filter**: files whose footers lack ra/dec statistics are
  always read. If a pixel's bounding box crosses RA=0, RA pruning is skipped for safety.
- **Sort keys** (`ids` in `CATALOGS`) must uniquely order rows within a partition — if a
  comparison reports differing columns *and* the id columns look fine, check for duplicate
  sort keys before suspecting the data (`left.duplicated(cfg["ids"]).any()`).
- Rows with NaN ra/dec (if any) are droppable at import — a level 1 row-count mismatch
  with clean aggregates elsewhere would point there; check `null_count` of ra/dec in the
  butler aggregates.
- Level 1 min/max compares parquet's *physical* statistics; both sides store the same
  types (the import copies values), so equality is exact, not approximate.
- All butler reads stream via `lsst.resources.ResourcePath`, same as the import itself.